<a href="https://colab.research.google.com/github/Harnoor001/UML501-TIET-MachineLearning/blob/main/UML501_Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================
# ASSIGNMENT 4 - UML501
# ==============================

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.datasets import load_iris

# ==============================
# Q1: RIDGE REGRESSION (FROM SCRATCH)
# ==============================

print("\n===== Q1: Ridge Regression from Scratch =====")

np.random.seed(42)
n_samples = 100

X_base = np.random.rand(n_samples, 1)
X = np.hstack([X_base + np.random.normal(0, 0.01, (n_samples, 1)) for _ in range(7)])

true_weights = np.array([2, -1.5, 3, 0.5, -2, 1, 4])
y = X @ true_weights + np.random.normal(0, 0.5, n_samples)

def ridge_gradient_descent(X, y, lr, lambda_, epochs=1000):
    n, m = X.shape
    weights = np.zeros(m)

    for _ in range(epochs):
        y_pred = X @ weights
        error = y_pred - y

        gradient = (2/n) * (X.T @ error) + 2 * lambda_ * weights
        weights -= lr * gradient

    return weights

learning_rates = [0.0001, 0.001, 0.01, 0.1]
lambdas = [1e-5, 1e-3, 0, 1, 10]

best_r2 = -np.inf

for lr in learning_rates:
    for lam in lambdas:
        weights = ridge_gradient_descent(X, y, lr, lam)
        y_pred = X @ weights

        r2 = r2_score(y, y_pred)
        cost = np.mean((y - y_pred)**2) + lam * np.sum(weights**2)

        print(f"LR: {lr}, Lambda: {lam}, Cost: {cost:.4f}, R2: {r2:.4f}")

        if r2 > best_r2:
            best_r2 = r2
            best_params = (lr, lam)

print("Best Parameters:", best_params)

# ==============================
# Q2: HITTERS DATASET
# ==============================

print("\n===== Q2: Hitters Dataset =====")

# Upload hitters.csv manually in Colab
try:
    df = pd.read_csv("Hitters.csv")
except:
    print("⚠️ Upload Hitters.csv manually in Colab!")
    df = None

if df is not None:
    df = df.dropna()
    df = pd.get_dummies(df, drop_first=True)

    X = df.drop("Salary", axis=1)
    y = df["Salary"]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    linear = LinearRegression()
    ridge = Ridge(alpha=0.5748)
    lasso = Lasso(alpha=0.5748)

    linear.fit(X_train, y_train)
    ridge.fit(X_train, y_train)
    lasso.fit(X_train, y_train)

    models = {"Linear": linear, "Ridge": ridge, "Lasso": lasso}

    for name, model in models.items():
        y_pred = model.predict(X_test)
        print(f"{name} R2 Score:", r2_score(y_test, y_pred))

# ==============================
# Q3: RIDGECV & LASSOCV
# ==============================

print("\n===== Q3: RidgeCV & LassoCV =====")

# Using synthetic dataset (since load_boston is removed)
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=100, n_features=10, noise=0.1)

ridge_cv = RidgeCV(alphas=[0.1, 1, 10])
ridge_cv.fit(X, y)

lasso_cv = LassoCV(alphas=[0.1, 1, 10])
lasso_cv.fit(X, y)

print("Best Alpha Ridge:", ridge_cv.alpha_)
print("Best Alpha Lasso:", lasso_cv.alpha_)

# ==============================
# Q4: MULTICLASS LOGISTIC REGRESSION (FROM SCRATCH)
# ==============================

print("\n===== Q4: Multiclass Logistic Regression (OvR) =====")

data = load_iris()
X = data.data
y = data.target

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_ovr(X, y, lr=0.01, epochs=1000):
    n_classes = len(np.unique(y))
    n_features = X.shape[1]

    weights = np.zeros((n_classes, n_features))

    for c in range(n_classes):
        y_binary = (y == c).astype(int)
        w = np.zeros(n_features)

        for _ in range(epochs):
            z = X @ w
            y_pred = sigmoid(z)

            gradient = X.T @ (y_pred - y_binary) / len(y)
            w -= lr * gradient

        weights[c] = w

    return weights

def predict_ovr(X, weights):
    probs = sigmoid(X @ weights.T)
    return np.argmax(probs, axis=1)

weights = train_ovr(X, y)
y_pred = predict_ovr(X, weights)

print("Accuracy:", accuracy_score(y, y_pred))

# ==============================
# END
# ==============================